In [23]:
pip install python-whois requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\User\Phishing_Detection_Using_Machine_Learning\venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=4>
  res = process_handler(cmd, _system_body)
C:\Users\User\Phishing_Detection_Using_Machine_Learning\venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=6>
  res = process_handler(cmd, _system_body)
C:\Users\User\Phishing_Detection_Using_Machine_Learning\venv\Lib\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=7>
  res = process_handler(cmd, _system_body)


In [24]:
python test_extractor.py

SyntaxError: invalid syntax (1181216790.py, line 1)

In [1]:
import pandas as pd

data_new = pd.read_csv("dataset_phishing.csv")
print(data_new.shape)
print(data_new['status'].value_counts())
print(data_new.isnull().sum().sum())

(11430, 89)
status
legitimate    5715
phishing      5715
Name: count, dtype: int64
0


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pickle
import os

# Prepare features and target
X_new = data_new.drop(columns=['url', 'status'])
Y_new = data_new['status']

# Split
X_train, X_test, Y_train, Y_test = train_test_split(X_new, Y_new, test_size=0.3, random_state=2)

# Train Random Forest
rf_new = RandomForestClassifier(n_estimators=100, random_state=2)
rf_new.fit(X_train, Y_train)

# Evaluate
predictions = rf_new.predict(X_test)
accuracy = accuracy_score(Y_test, predictions)
print(f"New Model Accuracy: {round(accuracy * 100, 2)}%")
print(classification_report(Y_test, predictions))

# Save new model
os.makedirs("model", exist_ok=True)
with open("model/phishing_model.pkl", "wb") as f:
    pickle.dump(rf_new, f)

# Save feature names for extractor
feature_names_new = list(X_new.columns)
with open("model/feature_names.pkl", "wb") as f:
    pickle.dump(feature_names_new, f)

print("New model saved successfully.")
print(f"Number of features: {len(feature_names_new)}")
print(feature_names_new)

New Model Accuracy: 96.3%
              precision    recall  f1-score   support

  legitimate       0.97      0.96      0.96      1747
    phishing       0.96      0.97      0.96      1682

    accuracy                           0.96      3429
   macro avg       0.96      0.96      0.96      3429
weighted avg       0.96      0.96      0.96      3429

New model saved successfully.
Number of features: 87
['length_url', 'length_hostname', 'ip', 'nb_dots', 'nb_hyphens', 'nb_at', 'nb_qm', 'nb_and', 'nb_or', 'nb_eq', 'nb_underscore', 'nb_tilde', 'nb_percent', 'nb_slash', 'nb_star', 'nb_colon', 'nb_comma', 'nb_semicolumn', 'nb_dollar', 'nb_space', 'nb_www', 'nb_com', 'nb_dslash', 'http_in_path', 'https_token', 'ratio_digits_url', 'ratio_digits_host', 'punycode', 'port', 'tld_in_path', 'tld_in_subdomain', 'abnormal_subdomain', 'nb_subdomains', 'prefix_suffix', 'random_domain', 'shortening_service', 'path_extension', 'nb_redirection', 'nb_external_redirection', 'length_words_raw', 'char_repeat'

In [2]:
import pandas as pd

data_new = pd.read_csv("dataset_phishing.csv")

# Check what the training data actually looks like for a few key features
print(data_new[['length_url', 'domain_age', 'nb_dots', 'ip', 'prefix_suffix']].describe())

# Also check what values legitimate sites have
print("\nLegitimate samples:")
print(data_new[data_new['status'] == 'legitimate'][['length_url', 'domain_age', 'web_traffic', 'page_rank']].describe())

# And phishing samples
print("\nPhishing samples:")
print(data_new[data_new['status'] == 'phishing'][['length_url', 'domain_age', 'web_traffic', 'page_rank']].describe())

         length_url    domain_age       nb_dots            ip  prefix_suffix
count  11430.000000  11430.000000  11430.000000  11430.000000   11430.000000
mean      61.126684   4062.543745      2.480752      0.150569       0.202450
std       55.297318   3107.784600      1.369686      0.357644       0.401843
min       12.000000    -12.000000      1.000000      0.000000       0.000000
25%       33.000000    972.250000      2.000000      0.000000       0.000000
50%       47.000000   3993.000000      2.000000      0.000000       0.000000
75%       71.000000   7026.750000      3.000000      0.000000       0.000000
max     1641.000000  12874.000000     24.000000      1.000000       1.000000

Legitimate samples:
        length_url    domain_age   web_traffic    page_rank
count  5715.000000   5715.000000  5.715000e+03  5715.000000
mean     47.381452   5093.938408  7.362518e+05     4.482415
std      27.862702   3101.132343  1.682119e+06     1.975390
min      12.000000     -2.000000  0.000000e+00

In [3]:
# Check how much these three features actually matter to the model
import pickle
import numpy as np

with open("model/phishing_model.pkl", "rb") as f:
    model = pickle.load(f)

feature_names = list(data_new.drop(columns=['url', 'status']).columns)
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

print("Top 15 most important features:")
for i in range(15):
    print(f"{feature_names[indices[i]]}: {round(importances[indices[i]] * 100, 2)}%")

Top 15 most important features:
google_index: 18.23%
page_rank: 10.01%
web_traffic: 8.06%
nb_hyperlinks: 7.95%
nb_www: 3.98%
ratio_extHyperlinks: 3.65%
domain_age: 3.11%
phish_hints: 2.96%
longest_word_path: 2.54%
ratio_intHyperlinks: 2.51%
ratio_digits_url: 2.09%
safe_anchor: 2.09%
length_url: 1.6%
domain_in_title: 1.43%
longest_words_raw: 1.41%


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pickle

# Drop unreliable features we can't extract live
drop_cols = ['url', 'status', 'google_index', 'page_rank', 'web_traffic', 'statistical_report']

X_new = data_new.drop(columns=drop_cols)
Y_new = data_new['status']

X_train, X_test, Y_train, Y_test = train_test_split(X_new, Y_new, test_size=0.3, random_state=2)

rf_new = RandomForestClassifier(n_estimators=100, random_state=2)
rf_new.fit(X_train, Y_train)

predictions = rf_new.predict(X_test)
accuracy = accuracy_score(Y_test, predictions)
print(f"New Model Accuracy: {round(accuracy * 100, 2)}%")
print(classification_report(Y_test, predictions))

# Save new model
with open("model/phishing_model.pkl", "wb") as f:
    pickle.dump(rf_new, f)

# Save new feature names
feature_names_new = list(X_new.columns)
with open("model/feature_names.pkl", "wb") as f:
    pickle.dump(feature_names_new, f)

print(f"\nModel saved. Features: {len(feature_names_new)}")
print(feature_names_new)

New Model Accuracy: 94.95%
              precision    recall  f1-score   support

  legitimate       0.95      0.95      0.95      1747
    phishing       0.95      0.95      0.95      1682

    accuracy                           0.95      3429
   macro avg       0.95      0.95      0.95      3429
weighted avg       0.95      0.95      0.95      3429


Model saved. Features: 83
['length_url', 'length_hostname', 'ip', 'nb_dots', 'nb_hyphens', 'nb_at', 'nb_qm', 'nb_and', 'nb_or', 'nb_eq', 'nb_underscore', 'nb_tilde', 'nb_percent', 'nb_slash', 'nb_star', 'nb_colon', 'nb_comma', 'nb_semicolumn', 'nb_dollar', 'nb_space', 'nb_www', 'nb_com', 'nb_dslash', 'http_in_path', 'https_token', 'ratio_digits_url', 'ratio_digits_host', 'punycode', 'port', 'tld_in_path', 'tld_in_subdomain', 'abnormal_subdomain', 'nb_subdomains', 'prefix_suffix', 'random_domain', 'shortening_service', 'path_extension', 'nb_redirection', 'nb_external_redirection', 'length_words_raw', 'char_repeat', 'shortest_words_raw', '

In [2]:
import pickle
import pandas as pd


data_new = pd.read_csv("dataset_phishing.csv")

with open("model/feature_names.pkl", "rb") as f:
    feature_names = pickle.load(f)

# What does a typical legitimate site look like
legitimate_median = data_new[data_new['status'] == 'legitimate'][feature_names].median()
phishing_median = data_new[data_new['status'] == 'phishing'][feature_names].median()

# Our google.com features
google_features = [17, 10, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0.0, 0.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 3, 2, 3, 3, 0, 6, 6, 0, 4.33, 4.5, 0.0, 0, 1, 0, 0, 0, 13, 0.6154, 0.0, 0.0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.0, 0, 0, 0, 1, 1, 1, 11322, 0, 1]

comparison = pd.DataFrame({
    'feature': feature_names,
    'google_value': google_features,
    'legitimate_median': legitimate_median.values,
    'phishing_median': phishing_median.values
})

# Show only features where google looks more like phishing than legitimate
comparison['closer_to_phishing'] = abs(comparison['google_value'] - comparison['phishing_median']) < abs(comparison['google_value'] - comparison['legitimate_median'])

print("Features where google.com looks like phishing:")
print(comparison[comparison['closer_to_phishing']][['feature', 'google_value', 'legitimate_median', 'phishing_median']])

Features where google.com looks like phishing:
                 feature  google_value  legitimate_median  phishing_median
20                nb_www        0.0000           1.000000         0.000000
40           char_repeat        2.0000           3.000000         2.000000
55         nb_hyperlinks       13.0000          86.000000        14.000000
56   ratio_intHyperlinks        0.6154           0.800000         0.538462
57   ratio_extHyperlinks        0.0000           0.155405         0.069930
61  ratio_extRedirection        0.0000           0.096154         0.000000
65      external_favicon        0.0000           1.000000         0.000000
66         links_in_tags        0.0000          66.666667        33.333333
68        ratio_intMedia        0.0000          63.157895         0.000000
73           safe_anchor        0.0000          42.857143         0.000000
81            domain_age        0.0000        5563.000000      2352.000000
